In [ ]:
pip install requests beautifulsoup4 lxml cloudscraper selenium

In [ ]:
pip install undetected_chromedriver

## Visit Oman Extraction

In [ ]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import json
import time
import pandas as pd

def setup_driver():
    options = uc.ChromeOptions()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    return uc.Chrome(options=options)

def get_all_property_links_selenium(max_pages=10):
    driver = setup_driver()
    all_links = set()

    for page in range(1, max_pages + 1):
        url = "https://vistaoman.com/properties/" if page == 1 else f"https://vistaoman.com/properties/?current_page={page}"
        print(f"[Page {page}] Loading: {url}")
        driver.get(url)
        time.sleep(4)
        soup = BeautifulSoup(driver.page_source, "html.parser")

        # Target only correct property links
        buttons = soup.select("div.mh-estate-vertical__buttons__single a[href]")
        for btn in buttons:
            href = btn.get("href", "")
            full_url = urljoin("https://vistaoman.com", href)
            if "Properties-for-sale-rent" in full_url:
                all_links.add(full_url)

        print(f"✔ Found {len(buttons)} links on page {page} — Total collected: {len(all_links)}")

    driver.quit()
    print(f"✅ Total {len(all_links)} unique property links collected.")
    return list(all_links)

def extract_full_page_content(driver, url):
    try:
        driver.get(url)
        time.sleep(3)
        soup = BeautifulSoup(driver.page_source, "html.parser")
        raw_text = soup.get_text(separator=' ', strip=True)

        return {
            "url": url,
            "raw_text": raw_text,
            "html": soup.prettify()
        }
    except Exception as e:
        print(f"❌ Failed to extract from {url}: {e}")
        return None

def scrape_vistaoman_raw(max_pages=10):
    links = get_all_property_links_selenium(max_pages)
    driver = setup_driver()
    results = []

    for i, link in enumerate(links):
        print(f"[{i+1}/{len(links)}] Extracting: {link}")
        prop = extract_full_page_content(driver, link)
        if prop:
            results.append(prop)
        time.sleep(1.5)

    driver.quit()

    # Save JSON
    with open("vistaoman_raw_data.json", "w", encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    # Save CSV (just url and text)
    df = pd.DataFrame(results)[["url", "raw_text"]]
    df.to_csv("vistaoman_raw_data.csv", index=False)
    print("✅ Done! Saved vistaoman_raw_data.json and vistaoman_raw_data.csv")

# Run the scraper
scrape_vistaoman_raw(max_pages=20)


In [ ]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import pandas as pd
import json
import time

# Load the CSV containing the URLs
df = pd.read_csv("vistaoman_raw_data.csv")  # Change if your filename differs
urls = df["url"].dropna().unique().tolist()

def setup_driver():
    options = uc.ChromeOptions()
    options.add_argument("--headless")  # Run in headless mode
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    return uc.Chrome(options=options)

def extract_property_page(driver, url):
    try:
        driver.get(url)
        time.sleep(3)  # Give time for the page to load
        soup = BeautifulSoup(driver.page_source, "html.parser")

        return {
            "url": url,
            "html": soup.prettify(),
            "raw_text": soup.get_text(separator=' ', strip=True)
        }
    except Exception as e:
        print(f"❌ Failed to scrape {url}: {e}")
        return None

def extract_all_pages(url_list):
    driver = setup_driver()
    results = []

    for i, url in enumerate(url_list, 1):
        print(f"[{i}/{len(url_list)}] Scraping: {url}")
        data = extract_property_page(driver, url)
        if data:
            results.append(data)
        time.sleep(0.7)  # Light delay between requests

    driver.quit()

    # Save to JSON
    with open("vistaoman_detailed_pages.json", "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    print(f"✅ Done. {len(results)} pages scraped and saved to vistaoman_detailed_pages.json")

# Run the extractor
extract_all_pages(urls)


In [48]:
import json
import csv
from bs4 import BeautifulSoup
import re
import pandas as pd

with open("vistaoman_raw_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []
all_features = set()

def extract_text(soup, label):
    """Extract label-based text (tries multiple ways)."""
    # 1) Try exact label: "Bedrooms:"
    match = soup.find(text=re.compile(rf"{label}\s*:", re.I))
    if match and match.parent:
        return match.parent.get_text(strip=True).split(":")[-1].strip()
    # 2) Try label without colon
    match2 = soup.find(text=re.compile(rf"{label}", re.I))
    if match2 and match2.parent:
        text = match2.parent.get_text(" ", strip=True)
        return text.replace(label, "").strip(": -")
    return ""

for entry in data:
    soup = BeautifulSoup(entry["html"], "html.parser")
    row = {}

    # Title
    row["title"] = soup.select_one("h1").get_text(strip=True) if soup.select_one("h1") else ""

    # Price (Fix: convert to string, not <re.Match object>)
    price_match = re.search(r"OMR\s?[\d,]+", entry.get("raw_text", ""), re.I)
    row["price"] = price_match.group(0) if price_match else extract_text(soup, "Price")

    # Other basic fields
    row["sale_rent"] = extract_text(soup, "Sale/Rent")
    row["bedrooms"] = extract_text(soup, "Bedrooms") or re.search(r"(\d+)\s*BR", row["title"], re.I).group(1) if re.search(r"(\d+)\s*BR", row["title"], re.I) else ""
    row["bathrooms"] = extract_text(soup, "Bathroom")
    row["ref_no"] = extract_text(soup, "Ref No")
    row["city"] = extract_text(soup, "City")

    # Contact info
    contact_block = soup.find(string=re.compile(r"@vistaoman\.com"))
    row["email"] = contact_block.strip() if contact_block else ""
    phone_match = re.search(r"\+968\d{7,9}", entry.get("raw_text", ""))
    row["phone"] = phone_match.group(0) if phone_match else ""

    # Features
    features = [
        "24/7 Security", "Parking", "Shared Gym", "Shared Pool", "Balcony",
        "Built-in Wardrobes", "Maid's Room", "Fully Equipped Kitchen", "Spacious Living Room"
    ]
    for feat in features:
        feat_present = re.search(re.escape(feat).replace("24/7", r"(24.?7)"), entry.get("raw_text", ""), re.I)
        row[feat] = 1 if feat_present else 0
        all_features.add(feat)

    rows.append(row)

# Save CSV
df = pd.DataFrame(rows)
df.to_csv("vistaoman_parsed_data.csv", index=False)
print(f"✅ Saved to vistaoman_parsed_data.csv with {len(rows)} listings.")


/tmp/ipykernel_30195/3657780777.py:16: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  match = soup.find(text=re.compile(rf"{label}\s*:", re.I))
/tmp/ipykernel_30195/3657780777.py:20: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  match2 = soup.find(text=re.compile(rf"{label}", re.I))


✅ Saved to vistaoman_parsed_data.csv with 179 listings.


In [49]:
import json
import re
from bs4 import BeautifulSoup
import pandas as pd

with open("vistaoman_raw_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []
dynamic_features = set()

def extract_jsonld(soup):
    """Extract JSON-LD structured data if available."""
    jsonld = soup.find("script", type="application/ld+json")
    if jsonld:
        try:
            return json.loads(jsonld.string)
        except:
            return {}
    return {}

def extract_from_title(title, key):
    """Extract fields from title when structured data is missing."""
    if key == "bedrooms":
        m = re.search(r"(\d+)\s*BR", title, re.I)
        return m.group(1) if m else ""
    if key == "city":
        m = re.search(r"in\s+([\w\s]+)$", title)
        return m.group(1).strip() if m else ""
    if key == "property_type":
        m = re.search(r"(Apartment|Villa|Penthouse|Office|Shop)", title, re.I)
        return m.group(1).title() if m else ""
    return ""

for entry in data:
    soup = BeautifulSoup(entry["html"], "html.parser")
    jsonld = extract_jsonld(soup)
    row = {}

    # Title
    row["title"] = soup.select_one("h1").get_text(strip=True) if soup.select_one("h1") else ""

    # Price
    price_match = re.search(r"OMR\s?[\d,]+", entry.get("raw_text", ""), re.I)
    row["price"] = price_match.group(0) if price_match else jsonld.get("offers", {}).get("price", "")

    # Bedrooms & Bathrooms
    row["bedrooms"] = jsonld.get("numberOfRooms", "") or extract_from_title(row["title"], "bedrooms")
    row["bathrooms"] = jsonld.get("numberOfBathroomsTotal", "")
    if not row["bathrooms"]:
        bath_match = re.search(r"(\d+)\s*(Bath|Bathroom)", entry.get("raw_text", ""), re.I)
        row["bathrooms"] = bath_match.group(1) if bath_match else ""

    # Location / City
    row["city"] = jsonld.get("address", {}).get("addressLocality", "") or extract_from_title(row["title"], "city")

    # Property type
    row["property_type"] = jsonld.get("@type", "") or extract_from_title(row["title"], "property_type")

    # Contact info
    contact_block = soup.find(string=re.compile(r"@vistaoman\.com"))
    row["email"] = contact_block.strip() if contact_block else ""
    phone_match = re.search(r"\+968\d{7,9}", entry.get("raw_text", ""))
    row["phone"] = phone_match.group(0) if phone_match else ""

    # Features (dynamic)
    features_section = soup.find_all(["li", "span"], text=True)
    for f in features_section:
        feat_text = f.get_text(strip=True)
        if len(feat_text.split()) <= 5 and not re.match(r"^\d", feat_text):
            dynamic_features.add(feat_text)
            row[feat_text] = 1 if feat_text.lower() in entry.get("raw_text", "").lower() else 0

    rows.append(row)

# Convert to DataFrame
df = pd.DataFrame(rows).fillna("")
df.to_csv("vistaoman_parsed_data_v2.csv", index=False)
print(f"✅ Saved to vistaoman_parsed_data_v2.csv with {len(rows)} listings & {len(dynamic_features)} features.")


/tmp/ipykernel_30195/920981449.py:67: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  features_section = soup.find_all(["li", "span"], text=True)


✅ Saved to vistaoman_parsed_data_v2.csv with 179 listings & 626 features.


## Tibiaan Extraction

In [ ]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import time
import json
import pandas as pd

def setup_driver():
    options = uc.ChromeOptions()
    # Do NOT use headless so you can interact
    return uc.Chrome(options=options)

def scrape_listings_from_page(driver):
    soup = BeautifulSoup(driver.page_source, "html.parser")
    listings = soup.select("div.listing-content")

    data = []
    for l in listings:
        try:
            title_tag = l.select_one("h4 a")
            title = title_tag.get_text(strip=True) if title_tag else ""
            href = title_tag.get("href", "") if title_tag else ""
            url = "https://tibiaan.com/" + href if href else ""
            raw_text = l.get_text(separator=" ", strip=True)

            data.append({
                "title": str(title),
                "url": str(url),
                "raw_text": str(raw_text)
            })
        except Exception as e:
            print(f"⚠️ Skipped a listing due to: {e}")
    return data

def scrape_tibiaan_after_filter():
    driver = setup_driver()
    driver.get("https://tibiaan.com/property-search")
    print("🔔 Please select filters (For Sale / For Rent) and click 'Update'")
    input("✅ Press ENTER once listings are visible to begin scraping...")

    all_data = []
    visited_pages = set()
    current_page = 1

    while True:
        time.sleep(3)  # let listings load
        print(f"📄 Scraping page {current_page}")

        # Scrape and append
        page_data = scrape_listings_from_page(driver)
        print(f"📦 Found {len(page_data)} listings")
        all_data.extend(page_data)

        # Parse pagination links
        soup = BeautifulSoup(driver.page_source, "html.parser")
        next_link = soup.select_one(f'a.update-page[data-page="{current_page + 1}"]')

        if next_link and (str(current_page + 1) not in visited_pages):
            try:
                print(f"➡️ Clicking to page {current_page + 1}")
                driver.execute_script("arguments[0].click();", next_link)
                visited_pages.add(str(current_page + 1))
                current_page += 1
                time.sleep(2)
            except Exception as e:
                print(f"⚠️ Failed to click page {current_page + 1}: {e}")
                break
        else:
            print("🚫 No more pages to visit.")
            break

    driver.quit()

    # Save results (JSON and CSV)
    with open("tibiaan_all_pages.json", "w", encoding="utf-8") as f:
        json.dump(all_data, f, indent=2, ensure_ascii=False)

    pd.DataFrame(all_data).to_csv("tibiaan_all_pages.csv", index=False)
    print(f"✅ Done. Total listings collected: {len(all_data)}")

# 🚀 Run the scraper
scrape_tibiaan_after_filter()


In [ ]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import pandas as pd
import json
import time

# Load the CSV containing the URLs
df = pd.read_csv("tibiaan_all_results.csv")  # Change if your filename differs
urls = df["url"].dropna().unique().tolist()

def setup_driver():
    options = uc.ChromeOptions()
    options.add_argument("--headless")  # Run in headless mode
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    return uc.Chrome(options=options)

def extract_property_page(driver, url):
    try:
        driver.get(url)
        time.sleep(3)  # Give time for the page to load
        soup = BeautifulSoup(driver.page_source, "html.parser")

        return {
            "url": url,
            "html": soup.prettify(),
            "raw_text": soup.get_text(separator=' ', strip=True)
        }
    except Exception as e:
        print(f"❌ Failed to scrape {url}: {e}")
        return None

def extract_all_pages(url_list):
    driver = setup_driver()
    results = []

    for i, url in enumerate(url_list, 1):
        print(f"[{i}/{len(url_list)}] Scraping: {url}")
        data = extract_property_page(driver, url)
        if data:
            results.append(data)
        time.sleep(0.7)  # Light delay between requests

    driver.quit()

    # Save to JSON
    with open("tibiaan_detailed_pages.json", "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    print(f"✅ Done. {len(results)} pages scraped and saved to tibiaan_detailed_pages.json")

# Run the extractor
extract_all_pages(urls)


In [ ]:
import json
import re
from bs4 import BeautifulSoup
import pandas as pd

INPUT_FILE = "tibiaan_detailed_pages.json"   # Change to your second file name
OUTPUT_FILE = "tibiaan_parsed_data_file2.csv"

FEATURES = [
    "24/7 Security", "Parking", "Shared Gym", "Shared Pool", "Balcony",
    "Built-in Wardrobes", "Maid's Room", "Fully Equipped Kitchen", "Spacious Living Room"
]

def extract_jsonld(soup):
    tag = soup.find("script", type="application/ld+json")
    if not tag:
        return {}
    try:
        data = json.loads(tag.string)
        if isinstance(data, dict) and "@graph" in data:
            for obj in data["@graph"]:
                if obj.get("@type") in ["RealEstateListing", "Product"]:
                    return obj
        return data
    except:
        return {}

def extract_text(soup, label):
    match = soup.find(text=re.compile(rf"{label}\s*:", re.I))
    if match and match.parent:
        return match.parent.get_text(strip=True).split(":")[-1].strip()
    return ""

def extract_from_title(title, key):
    if key == "bedrooms":
        m = re.search(r"(\d+)\s*BR", title, re.I)
        return m.group(1) if m else ""
    if key == "city":
        m = re.search(r"in\s+([\w\s]+)$", title)
        return m.group(1).strip() if m else ""
    return ""

rows = []

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

for entry in data:
    soup = BeautifulSoup(entry.get("html", ""), "html.parser")
    jsonld = extract_jsonld(soup)
    raw_text = entry.get("raw_text", "")

    # ✅ Skip error pages
    title_tag = soup.select_one("h1")
    if not title_tag or "error" in title_tag.get_text(strip=True).lower():
        continue

    row = {}
    title = title_tag.get_text(strip=True)
    row["title"] = title

    # Price
    price_match = re.search(r"OMR\s?[\d,]+", raw_text, re.I)
    row["price"] = price_match.group(0) if price_match else jsonld.get("offers", {}).get("price", "")

    # Sale / Rent
    if "rent" in entry.get("url", "").lower():
        row["sale_rent"] = "Rent"
    elif "sale" in entry.get("url", "").lower():
        row["sale_rent"] = "Sale"
    else:
        row["sale_rent"] = ""

    # Bedrooms & Bathrooms
    row["bedrooms"] = jsonld.get("numberOfRooms", "") or extract_text(soup, "Bedrooms") or extract_from_title(title, "bedrooms")
    row["bathrooms"] = jsonld.get("numberOfBathroomsTotal", "") or extract_text(soup, "Bathroom")

    # Ref No
    row["ref_no"] = extract_text(soup, "Ref No")

    # City
    row["city"] = jsonld.get("address", {}).get("addressLocality", "") or extract_text(soup, "City") or extract_from_title(title, "city")

    # Contact Info
    contact_block = soup.find(string=re.compile(r"@vistaoman\.com"))
    row["email"] = contact_block.strip() if contact_block else ""
    phone_match = re.search(r"\+968\d{7,9}", raw_text)
    row["phone"] = phone_match.group(0) if phone_match else ""

    # Features
    for feat in FEATURES:
        feat_present = re.search(re.escape(feat).replace("24/7", r"(24.?7)"), raw_text, re.I)
        row[feat] = 1 if feat_present else 0

    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_FILE, index=False)
print(f"✅ Saved to {OUTPUT_FILE} with {len(rows)} valid listings (error pages skipped).")


## Data Cleaning

In [58]:
import pandas as pd
import numpy as np

# ---------- 1. Load Data ----------
df = pd.read_csv("vistaoman_parsed_data_v2.csv")

# ---------- 2. Define Columns ----------
IMPORTANT_FEATURES = [
    "24/7 Security", "Parking", "Shared Gym", "Shared Pool", "Balcony",
    "Built-in Wardrobes", "Maid's Room", "Fully Equipped Kitchen", "Spacious Living Room"
]

main_cols = [
    "title", "price", "bedrooms", "bathrooms", "city",
    "property_type", "email", "phone"
]

# Other features (for scoring only, will be dropped later)
other_features = [col for col in df.columns if col not in main_cols + IMPORTANT_FEATURES]

# ---------- 3. Clean Numeric Fields ----------
df["bedrooms"] = pd.to_numeric(df["bedrooms"], errors="coerce")
df["bathrooms"] = pd.to_numeric(df["bathrooms"], errors="coerce")

df["bedrooms"] = df["bedrooms"].fillna(round(df["bedrooms"].mean(skipna=True), 1))
df["bathrooms"] = df["bathrooms"].fillna(round(df["bathrooms"].mean(skipna=True), 1))

# ---------- 4. Clean City ----------
df["city"] = df["city"].fillna("Unknown")
df["city"] = df["city"].replace("", "Unknown")

# ---------- 5. Convert Important Features to True/False ----------
for col in IMPORTANT_FEATURES:
    if col in df.columns:
        df[col] = df[col].apply(
            lambda x: True if pd.notna(x) and float(str(x)) == 1.0 else False
        )
    else:
        df[col] = False  # If the column doesn't exist, default to False

# ---------- 6. Feature Score (Important + Other Features) ----------
score_df = df[IMPORTANT_FEATURES].applymap(lambda x: 1 if x is True else 0)

if other_features:
    other_score_df = df[other_features].applymap(
        lambda x: 1 if pd.notna(x) and float(str(x)) == 1.0 else 0
    )
    df["feature_score"] = score_df.sum(axis=1) + other_score_df.sum(axis=1)
else:
    df["feature_score"] = score_df.sum(axis=1)

# ---------- 7. Keep Only Required Columns ----------
final_df = df[main_cols + IMPORTANT_FEATURES + ["feature_score"]]

# ---------- 8. Save ----------
final_df.to_csv("cleaned_data.csv", index=False)
print(f"✅ Cleaned & saved to cleaned_data.csv with {len(final_df)} rows.")


✅ Cleaned & saved to cleaned_data.csv with 179 rows.


/tmp/ipykernel_30195/2812109224.py:42: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  score_df = df[IMPORTANT_FEATURES].applymap(lambda x: 1 if x is True else 0)
/tmp/ipykernel_30195/2812109224.py:45: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  other_score_df = df[other_features].applymap(


## Feature Engineering and Scaling

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder

# ---------- 1. Load Cleaned CSV ----------
df = pd.read_csv("cleaned_data.csv")

# Ensure numeric types
df["price"] = pd.to_numeric(df["price"].replace('[^0-9.]', '', regex=True), errors="coerce").fillna(0)
df["bedrooms"] = pd.to_numeric(df["bedrooms"], errors="coerce")
df["bathrooms"] = pd.to_numeric(df["bathrooms"], errors="coerce")

# ---------- 2. New Features ----------
# Total rooms
df["total_rooms"] = df["bedrooms"] + df["bathrooms"]

# Price per room (avoid division by zero)
df["price_per_room"] = df.apply(
    lambda x: x["price"] / x["total_rooms"] if x["total_rooms"] > 0 else 0, axis=1
)

# Encode categorical values
le_city = LabelEncoder()
le_type = LabelEncoder()

df["city_encoded"] = le_city.fit_transform(df["city"].astype(str))
df["property_type_encoded"] = le_type.fit_transform(df["property_type"].astype(str))

# ---------- 3. Feature Scaling ----------
scaler_minmax = MinMaxScaler()
scaler_std = StandardScaler()

# Numeric columns to scale (price, price_per_room, total_rooms, feature_score)
scale_cols = ["price", "price_per_room", "total_rooms", "feature_score"]

df_minmax = pd.DataFrame(
    scaler_minmax.fit_transform(df[scale_cols]),
    columns=[f"{c}_minmax" for c in scale_cols]
)
df_std = pd.DataFrame(
    scaler_std.fit_transform(df[scale_cols]),
    columns=[f"{c}_std" for c in scale_cols]
)

# Concatenate scaled columns
df = pd.concat([df, df_minmax, df_std], axis=1)

# ---------- 4. Save ----------
df.to_csv("featured_data.csv", index=False)
print(f"✅ Model-ready CSV saved: featured_data.csv ({len(df)} rows)")


✅ Model-ready CSV saved: vistaoman_parsed_data_model_ready.csv (179 rows)
